In [ ]:
!pip install -q --no-cache-dir "vllm==0.19.1"
!pip install -q -U transformers huggingface_hub
!pip install -q "protobuf==6.33.6"
!pip install -q faiss-cpu

In [ ]:
# ============================================================
# tmp4.csv 기준
# project_id → project_description 연결
# company_patent_li 중복 제거
# BGE-M3 + FAISS Retrieval
# Qwen3.5 최종 특허 선택
#
# 저장 방식
# ------------------------------------------------------------
# - Google Drive CSV 하나만 사용
#
# - 저장 컬럼은 아래 10개만:
#
#   company_id
#   company_name
#   project_id
#   project_name
#   project_description
#   company_patent_original_count
#   company_patent_count
#   duplicate_removed_count
#   patent_less_than_5
#   rag_company_patent
#
# - 매 행 처리 완료:
#       위 10개 컬럼만 Google Drive CSV에 즉시 저장
#
# - 중단 후 재실행:
#       기존 결과 CSV를 읽어서
#       company_id + project_id 기준으로
#       rag_company_patent 복원
#
#       rag_company_patent 값이 있는 행은 SKIP
# ============================================================


# ============================================================
# 0. import / 환경 설정
# ============================================================

import os

os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

import ast
import json
import re
import time
import gc

import numpy as np
import pandas as pd
import torch
import faiss

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams


# ============================================================
# 1. 파일 경로
# ============================================================

TMP4_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/kisti/data/tmp4.csv"
)


EXP_RESULT_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/kisti/result/"
    "tmp4_qwen_comANDpro_exp_MoE.csv"
)


DRIVE_OUTPUT_PATH = (
    "/content/drive/MyDrive/Colab Notebooks/kisti/result/"
    "tmp4_qwen35_RAG_patent_over3.csv"
)


# ============================================================
# 2. 설정
# ============================================================

TOP_K = 20

MIN_PATENT_THRESHOLD = 5

MIN_SELECT_COUNT = 3


EMBED_MODEL_NAME = (
    "BAAI/bge-m3"
)


QWEN_MODEL_NAME = (
    "Qwen/Qwen3.5-35B-A3B-GPTQ-Int4"
)


# ============================================================
# 2-1. 실제 CSV 저장 컬럼
# ============================================================

SAVE_COLS = [

    "company_id",
    "company_name",

    "project_id",
    "project_name",
    "project_description",

    "company_patent_original_count",
    "company_patent_count",
    "duplicate_removed_count",
    "patent_less_than_5",

    "rag_company_patent"
]


# ============================================================
# 3. 공통 함수
# ============================================================

def clean_text(x):

    if x is None:
        return ""

    try:

        if pd.isna(x):
            return ""

    except Exception:
        pass


    x = str(x).strip()


    if x.lower() in {
        "nan",
        "none",
        "null"
    }:

        return ""


    return x


def normalize_project_id(x):

    x = clean_text(x)


    if not x:
        return ""


    if x.endswith(".0"):

        try:

            return str(
                int(
                    float(x)
                )
            )

        except Exception:
            pass


    return x


def normalize_company_id(x):

    x = clean_text(x)


    if not x:
        return ""


    if x.endswith(".0"):

        try:

            return str(
                int(
                    float(x)
                )
            )

        except Exception:
            pass


    return x


def normalize_patent_title(x):

    x = clean_text(x)


    x = re.sub(
        r"\s+",
        " ",
        x
    ).strip().lower()


    return x


def ensure_list(x):

    if isinstance(x, list):
        return x


    if isinstance(x, np.ndarray):
        return x.tolist()


    if x is None:
        return []


    try:

        if pd.isna(x):
            return []

    except Exception:
        pass


    if isinstance(x, str):

        x = x.strip()


        if not x:
            return []


        try:

            parsed = ast.literal_eval(x)


            if isinstance(parsed, list):
                return parsed


            return [parsed]


        except Exception:

            return []


    return [x]


# ============================================================
# 4. 데이터 로드
# ============================================================

tmp = pd.read_csv(
    TMP4_PATH
)


exp_desc = pd.read_csv(
    EXP_RESULT_PATH
)


print(
    "tmp4 shape:",
    tmp.shape
)


print(
    "exp_result shape:",
    exp_desc.shape
)


# ============================================================
# 5. 필요한 컬럼 확인
# ============================================================

required_tmp_cols = [

    "company_id",
    "company_name",
    "project_id",
    "project_name",
    "company_patent_li"
]


required_exp_cols = [

    "project_id",
    "project_description"
]


for col in required_tmp_cols:

    if col not in tmp.columns:

        raise ValueError(
            f"tmp4.csv에 필요한 컬럼이 없습니다: {col}"
        )


for col in required_exp_cols:

    if col not in exp_desc.columns:

        raise ValueError(
            f"EXP_RESULT_PATH에 필요한 컬럼이 없습니다: {col}"
        )


# ============================================================
# 6. project_id 정규화
# ============================================================

tmp[
    "_project_id"
] = (
    tmp[
        "project_id"
    ]
    .apply(
        normalize_project_id
    )
)


exp_desc[
    "_project_id"
] = (
    exp_desc[
        "project_id"
    ]
    .apply(
        normalize_project_id
    )
)


tmp[
    "company_id"
] = (
    tmp[
        "company_id"
    ]
    .apply(
        normalize_company_id
    )
)


# ============================================================
# 7. project_description master 생성
# ============================================================

project_desc_check = (

    exp_desc[
        [
            "_project_id",
            "project_description"
        ]
    ]
    .copy()
)


project_desc_check[
    "project_description"
] = (
    project_desc_check[
        "project_description"
    ]
    .apply(
        clean_text
    )
)


project_desc_check = (
    project_desc_check[
        project_desc_check[
            "_project_id"
        ] != ""
    ]
    .copy()
)


# 동일 project_id에 여러 description이 있으면
# 가장 긴 description 사용
project_description_rows = []


for project_id, block in (
    project_desc_check.groupby(
        "_project_id",
        sort=False
    )
):


    descriptions = [

        clean_text(x)

        for x in block[
            "project_description"
        ].tolist()

        if clean_text(x)
    ]


    if descriptions:

        project_description = max(
            descriptions,
            key=len
        )

    else:

        project_description = ""


    project_description_rows.append(
        {

            "_project_id":
                project_id,

            "project_description":
                project_description
        }
    )


project_description_master = (
    pd.DataFrame(
        project_description_rows
    )
)


# ============================================================
# 8. tmp4에 project_description 연결
# ============================================================

if (
    "project_description"
    in tmp.columns
):

    tmp = tmp.drop(
        columns=[
            "project_description"
        ]
    )


tmp = tmp.merge(

    project_description_master,

    on="_project_id",

    how="left",

    validate="many_to_one"
)


print(
    "\nproject_description merge 후 tmp shape:",
    tmp.shape
)


# ============================================================
# 9. project_description 매칭 확인
# ============================================================

project_match_fail = (

    tmp[
        "project_description"
    ]
    .isna()

    |

    (
        tmp[
            "project_description"
        ]
        .fillna("")
        .astype(str)
        .str.strip()
        == ""
    )
)


print(
    "project_description 매칭 실패:",
    int(
        project_match_fail.sum()
    )
)


# ============================================================
# 10. 각 행 company_patent_li 내부 중복 제거
# ============================================================

def deduplicate_patents(
    patent_value
):

    patent_list = ensure_list(
        patent_value
    )


    unique_patents = {}


    for patent_info in patent_list:


        if not isinstance(
            patent_info,
            dict
        ):

            continue


        patent = clean_text(
            patent_info.get(
                "patent",
                ""
            )
        )


        abstract = clean_text(
            patent_info.get(
                "abstract",
                ""
            )
        )


        if (
            not patent
            and
            not abstract
        ):

            continue


        patent_key = normalize_patent_title(
            patent
        )


        if not patent_key:

            abstract_key = re.sub(
                r"\s+",
                " ",
                abstract
            ).strip().lower()


            patent_key = (
                "__abstract__"
                +
                abstract_key
            )


        current = {

            "patent":
                patent,

            "abstract":
                abstract
        }


        if (
            patent_key
            not in unique_patents
        ):

            unique_patents[
                patent_key
            ] = current


        else:

            existing_abstract = clean_text(
                unique_patents[
                    patent_key
                ].get(
                    "abstract",
                    ""
                )
            )


            if (
                len(abstract)
                >
                len(existing_abstract)
            ):

                unique_patents[
                    patent_key
                ] = current


    return list(
        unique_patents.values()
    )


tqdm.pandas(
    desc="행별 특허 중복 제거"
)


tmp[
    "company_patent_li_unique"
] = (
    tmp[
        "company_patent_li"
    ]
    .progress_apply(
        deduplicate_patents
    )
)


# ============================================================
# 11. 특허 수 계산
# ============================================================

tmp[
    "company_patent_count"
] = (
    tmp[
        "company_patent_li_unique"
    ]
    .apply(len)
)


tmp[
    "patent_less_than_5"
] = (
    tmp[
        "company_patent_count"
    ]
    <
    MIN_PATENT_THRESHOLD
)


tmp[
    "company_patent_original_count"
] = (
    tmp[
        "company_patent_li"
    ]
    .apply(
        lambda x:
            len(
                ensure_list(x)
            )
    )
)


tmp[
    "duplicate_removed_count"
] = (
    tmp[
        "company_patent_original_count"
    ]
    -
    tmp[
        "company_patent_count"
    ]
)


# ============================================================
# 12. Embedding용 특허 문서 생성
# ============================================================

def make_patent_document(
    patent_info
):


    patent = clean_text(
        patent_info.get(
            "patent",
            ""
        )
    )


    abstract = clean_text(
        patent_info.get(
            "abstract",
            ""
        )
    )


    if abstract:

        return (
            f"특허명: {patent}\n"
            f"특허초록: {abstract}"
        )


    return (
        f"특허명: {patent}"
    )


# ============================================================
# 13. 전체 embedding 대상 수집
# ============================================================

all_patent_documents = []


for patent_list in tmp[
    "company_patent_li_unique"
]:

    for patent_info in patent_list:

        document = make_patent_document(
            patent_info
        )


        if document:

            all_patent_documents.append(
                document
            )


unique_patent_documents = list(
    dict.fromkeys(
        all_patent_documents
    )
)


unique_project_descriptions = [

    clean_text(x)

    for x in tmp[
        "project_description"
    ].tolist()

    if clean_text(x)
]


unique_project_descriptions = list(
    dict.fromkeys(
        unique_project_descriptions
    )
)


print(
    "\n고유 patent document:",
    len(
        unique_patent_documents
    )
)


print(
    "고유 project_description:",
    len(
        unique_project_descriptions
    )
)


# ============================================================
# 14. BGE-M3 로드
# ============================================================

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


embed_model = SentenceTransformer(
    EMBED_MODEL_NAME,
    device=DEVICE
)


# ============================================================
# 15. 특허 embedding
# ============================================================

patent_embeddings = embed_model.encode(

    unique_patent_documents,

    batch_size=64,

    show_progress_bar=True,

    normalize_embeddings=True
)


patent_embeddings = np.asarray(
    patent_embeddings,
    dtype="float32"
)


patent_embedding_map = {

    document:
        patent_embeddings[i]

    for i, document in enumerate(
        unique_patent_documents
    )
}


# ============================================================
# 16. project_description embedding
# ============================================================

project_embeddings = embed_model.encode(

    unique_project_descriptions,

    batch_size=64,

    show_progress_bar=True,

    normalize_embeddings=True
)


project_embeddings = np.asarray(
    project_embeddings,
    dtype="float32"
)


project_embedding_map = {

    description:
        project_embeddings[i]

    for i, description in enumerate(
        unique_project_descriptions
    )
}


# ============================================================
# 17. BGE-M3 GPU 해제
# ============================================================

embed_model.to(
    "cpu"
)


del embed_model


gc.collect()

torch.cuda.empty_cache()


# ============================================================
# 18. FAISS Retrieval
# ============================================================

def retrieve_patents_with_faiss(
    project_description,
    patent_list,
    top_k=20
):


    project_description = clean_text(
        project_description
    )


    if not project_description:
        return []


    if not isinstance(
        patent_list,
        list
    ):
        return []


    if len(patent_list) == 0:
        return []


    documents = []

    embeddings = []


    for patent_info in patent_list:


        document = make_patent_document(
            patent_info
        )


        embedding = patent_embedding_map.get(
            document
        )


        if embedding is None:
            continue


        documents.append(
            patent_info
        )


        embeddings.append(
            embedding
        )


    if len(embeddings) == 0:
        return []


    embeddings = np.asarray(
        embeddings,
        dtype="float32"
    )


    query_embedding = project_embedding_map.get(
        project_description
    )


    if query_embedding is None:
        return []


    query_embedding = np.asarray(
        [
            query_embedding
        ],
        dtype="float32"
    )


    dimension = embeddings.shape[1]


    index = faiss.IndexFlatIP(
        dimension
    )


    index.add(
        embeddings
    )


    search_k = min(
        top_k,
        len(embeddings)
    )


    scores, indices = index.search(
        query_embedding,
        search_k
    )


    scores = scores[0]

    indices = indices[0]


    candidates = []


    for candidate_index, (
        patent_idx,
        retrieval_score
    ) in enumerate(
        zip(
            indices,
            scores
        )
    ):


        if patent_idx < 0:
            continue


        patent_info = documents[
            patent_idx
        ]


        candidates.append(
            {

                "candidate_index":
                    candidate_index,

                "patent":
                    clean_text(
                        patent_info.get(
                            "patent",
                            ""
                        )
                    ),

                "abstract":
                    clean_text(
                        patent_info.get(
                            "abstract",
                            ""
                        )
                    ),

                "retrieval_score":
                    float(
                        retrieval_score
                    )
            }
        )


    return candidates


# ============================================================
# 19. Qwen 모델 로드
# ============================================================

HF_TOKEN = os.environ.get(
    "HF_TOKEN"
)


llm_kwargs = {

    "model":
        QWEN_MODEL_NAME,

    "trust_remote_code":
        True,

    "tensor_parallel_size":
        1
}


if HF_TOKEN:

    llm_kwargs[
        "hf_token"
    ] = HF_TOKEN


llm = LLM(
    **llm_kwargs
)


qwen_tok = AutoTokenizer.from_pretrained(

    QWEN_MODEL_NAME,

    trust_remote_code=True,

    token=HF_TOKEN
)


# ============================================================
# 20. Qwen 호출
# ============================================================

@torch.inference_mode()
def ask_qwen(
    prompt,
    min_select_count=0,
    max_new_tokens=2048
):


    system_prompt = (
        "당신은 국가 R&D 과제와 기업 특허 간의 "
        "기술적 연관성을 분석하는 전문가입니다. "
        "과제 설명과 특허명 및 특허초록의 자연어 내용을 "
        "직접 비교하여 관련 특허를 선택하세요. "
        "필요한 분석과 판단은 내부적으로 수행하세요. "
        "중간 추론 과정은 출력하지 마세요. "
        "Thinking Process, Analysis, Reasoning, "
        "후보별 검토 과정은 절대 출력하지 마세요. "
        "최종 JSON 배열만 출력하세요."
    )


    messages = [

        {
            "role":
                "system",

            "content":
                system_prompt
        },

        {
            "role":
                "user",

            "content":
                prompt
        }
    ]


    text = qwen_tok.apply_chat_template(

        messages,

        tokenize=False,

        add_generation_prompt=True
    )


    json_schema = {

        "type":
            "array",

        "items": {

            "type":
                "object",

            "properties": {

                "candidate_index": {

                    "type":
                        "integer",

                    "minimum":
                        0
                },

                "score": {

                    "type":
                        "integer",

                    "minimum":
                        0,

                    "maximum":
                        100
                },

                "reason": {

                    "type":
                        "string"
                }
            },

            "required": [

                "candidate_index",
                "score",
                "reason"
            ],

            "additionalProperties":
                False
        }
    }


    if min_select_count > 0:

        json_schema[
            "minItems"
        ] = min_select_count


    structured_outputs = (
        StructuredOutputsParams(
            json=json_schema
        )
    )


    params = SamplingParams(

        temperature=0.0,

        top_p=1.0,

        max_tokens=
            max_new_tokens,

        stop=[
            "<|im_end|>"
        ],

        structured_outputs=
            structured_outputs
    )


    outputs = llm.generate(
        [text],
        params
    )


    output = (
        outputs[0]
        .outputs[0]
    )


    response = output.text.strip()


    finish_reason = getattr(
        output,
        "finish_reason",
        None
    )


    if finish_reason == "length":

        print(
            "\n[경고] "
            "Qwen 출력이 max_tokens에 도달했습니다."
        )


    if "</think>" in response:

        response = response.split(
            "</think>",
            1
        )[1].strip()


    response = re.sub(
        r"^```json\s*",
        "",
        response,
        flags=re.IGNORECASE
    )


    response = re.sub(
        r"\s*```$",
        "",
        response
    ).strip()


    return response


# ============================================================
# 21. Qwen Prompt
# ============================================================

def rerank_patents_with_qwen(
    project_description,
    candidates,
    company_patent_count
):


    if not candidates:
        return "[]"


    if (
        company_patent_count
        >=
        MIN_PATENT_THRESHOLD
    ):


        min_select_count = (
            MIN_SELECT_COUNT
        )


        selection_rule = f"""
중복 제거 후 이 행의 company_patent_li에는
총 {company_patent_count}개의 서로 다른 특허가 있습니다.

최종 결과에는 반드시 최소 {MIN_SELECT_COUNT}개의
서로 다른 특허를 선택하세요.

직접적으로 관련성이 높은 특허가
{MIN_SELECT_COUNT}개보다 적더라도,
남은 후보 중 과제와 상대적으로 가장 기술적으로 가까운
특허를 추가하여 반드시 최소 {MIN_SELECT_COUNT}개를 선택하세요.

동일한 candidate_index를 중복 선택하지 마세요.

관련성이 높은 특허가 더 많다면
{MIN_SELECT_COUNT}개보다 많이 선택해도 됩니다.
"""


    else:


        min_select_count = 0


        selection_rule = f"""
중복 제거 후 이 행의 company_patent_li에는
총 {company_patent_count}개의 서로 다른 특허가 있습니다.

특허 수가 {MIN_PATENT_THRESHOLD}개 미만이므로
최소 선택 개수를 강제하지 않습니다.

과제와 자연어 기술 내용상 관련성이 있다고 판단되는
특허를 선택하세요.

관련 특허가 없다면 빈 배열 []을 반환할 수 있습니다.
"""


    candidate_texts = []


    for candidate in candidates:


        candidate_index = (
            candidate[
                "candidate_index"
            ]
        )


        patent = clean_text(
            candidate[
                "patent"
            ]
        )


        abstract = clean_text(
            candidate[
                "abstract"
            ]
        )


        candidate_text = f"""
[후보 {candidate_index}]

특허명:
{patent}

특허초록:
{abstract if abstract else "없음"}
""".strip()


        candidate_texts.append(
            candidate_text
        )


    patent_context = (
        "\n\n"
        "========================================"
        "\n\n"
    ).join(
        candidate_texts
    )


    prompt = f"""
아래 R&D 과제 설명과 특허 후보들을 비교하세요.

특허 후보들은 현재 tmp4.csv 행의
company_patent_li에서 직접 가져온 특허들입니다.

embedding 기반 Retrieval을 통해 후보가 선정되었지만,
최종 판단은 반드시 자연어 기술 내용만 보고 수행하세요.


==================================================
[과제 설명]
==================================================

{project_description}


==================================================
[특허 후보]
==================================================

{patent_context}


==================================================
[판단 기준]
==================================================

다음 정보만 사용하세요.

- 과제 설명
- 특허명
- 특허초록

다음 요소의 기술적 연결성을 종합적으로 판단하세요.

- 핵심 기술
- 제품
- 소재
- 부품
- 제조 공정
- 생산 방법
- 장치
- 시스템
- 알고리즘
- 소프트웨어
- 기술적 기능
- 활용 목적

단순 키워드 일치만으로 판단하지 마세요.

같은 산업이라는 이유만으로 선택하지 마세요.

retrieval score,
embedding similarity,
cosine similarity 값은
최종 판단 기준으로 사용하지 마세요.


==================================================
[선택 개수]
==================================================

{selection_rule}


==================================================
[출력 규칙]
==================================================

필요한 비교와 판단은 내부적으로 수행하세요.

Thinking Process를 출력하지 마세요.

Analysis나 Reasoning 과정을 출력하지 마세요.

최종 JSON 배열만 출력하세요.

JSON 앞뒤에 다른 문장을 출력하지 마세요.

첫 문자는 반드시 [
마지막 문자는 반드시 ] 이어야 합니다.

candidate_index는 반드시 실제 후보 번호만 사용하세요.

score는 0~100 사이 정수입니다.
"""


    return ask_qwen(

        prompt,

        min_select_count=
            min_select_count
    )


# ============================================================
# 22. JSON parsing
# ============================================================

def parse_qwen_json(
    response
):


    if response is None:
        return []


    response = str(
        response
    ).strip()


    try:

        parsed = json.loads(
            response
        )


        if isinstance(
            parsed,
            list
        ):

            return parsed


    except Exception:

        pass


    match = re.search(
        r"\[.*\]",
        response,
        flags=re.DOTALL
    )


    if match:

        try:

            parsed = json.loads(
                match.group()
            )


            if isinstance(
                parsed,
                list
            ):

                return parsed


        except Exception:

            pass


    print(
        "\n[JSON parsing 실패]"
    )


    print(
        response[:2000]
    )


    return []


# ============================================================
# 23. Qwen 결과 → 실제 특허 연결
# ============================================================

def build_selected_patents(
    candidates,
    llm_result,
    company_id,
    company_name
):


    candidate_map = {

        int(
            x[
                "candidate_index"
            ]
        ):
            x

        for x in candidates
    }


    selected = []

    used_indices = set()

    used_patent_titles = set()


    for item in llm_result:


        if not isinstance(
            item,
            dict
        ):

            continue


        try:

            candidate_index = int(
                item.get(
                    "candidate_index"
                )
            )

        except Exception:

            continue


        if (
            candidate_index
            not in candidate_map
        ):

            continue


        if (
            candidate_index
            in used_indices
        ):

            continue


        candidate = candidate_map[
            candidate_index
        ]


        patent = clean_text(
            candidate.get(
                "patent"
            )
        )


        patent_key = normalize_patent_title(
            patent
        )


        if (
            patent_key
            and
            patent_key in used_patent_titles
        ):

            continue


        selected.append(
            {

                "company_id":
                    company_id,

                "company_name":
                    company_name,

                "patent":
                    patent,

                "abstract":
                    candidate.get(
                        "abstract",
                        ""
                    ),

                "rag_score":
                    item.get(
                        "score"
                    ),

                "rag_reason":
                    item.get(
                        "reason"
                    )
            }
        )


        used_indices.add(
            candidate_index
        )


        if patent_key:

            used_patent_titles.add(
                patent_key
            )


    return selected


# ============================================================
# 24. 결과 컬럼 준비
# ============================================================

if (
    "rag_company_patent"
    not in tmp.columns
):

    tmp[
        "rag_company_patent"
    ] = ""


# ============================================================
# 25. 처리 여부
# ============================================================

def is_already_processed(x):

    x = clean_text(
        x
    )


    if not x:
        return False


    # []도 이미 Qwen까지 처리한 결과
    return True


# ============================================================
# 26. 기존 Drive 결과가 있으면 이어서 처리
#
# 결과 파일에는 10개 컬럼만 있어도 됨
#
# company_id + project_id로 기존 rag 결과 복원
# ============================================================

if os.path.exists(
    DRIVE_OUTPUT_PATH
):


    print(
        "\n기존 Drive 결과 파일 발견:"
    )

    print(
        DRIVE_OUTPUT_PATH
    )


    previous_result = pd.read_csv(
        DRIVE_OUTPUT_PATH
    )


    # 필요한 최소 컬럼 확인
    resume_required_cols = [
        "company_id",
        "project_id",
        "rag_company_patent"
    ]


    missing_resume_cols = [

        col

        for col in resume_required_cols

        if col not in previous_result.columns
    ]


    if missing_resume_cols:

        print(
            "[주의] 기존 결과에 이어하기 필수 컬럼이 없습니다:"
        )

        print(
            missing_resume_cols
        )


    else:


        previous_result[
            "_resume_company_id"
        ] = (
            previous_result[
                "company_id"
            ]
            .apply(
                normalize_company_id
            )
        )


        previous_result[
            "_resume_project_id"
        ] = (
            previous_result[
                "project_id"
            ]
            .apply(
                normalize_project_id
            )
        )


        tmp[
            "_resume_company_id"
        ] = (
            tmp[
                "company_id"
            ]
            .apply(
                normalize_company_id
            )
        )


        tmp[
            "_resume_project_id"
        ] = (
            tmp[
                "project_id"
            ]
            .apply(
                normalize_project_id
            )
        )


        previous_result[
            "_resume_processed"
        ] = (
            previous_result[
                "rag_company_patent"
            ]
            .apply(
                is_already_processed
            )
        )


        previous_done = (
            previous_result[
                previous_result[
                    "_resume_processed"
                ]
            ]
            [
                [
                    "_resume_company_id",
                    "_resume_project_id",
                    "rag_company_patent"
                ]
            ]
            .copy()
        )


        previous_done = (
            previous_done
            .drop_duplicates(
                subset=[
                    "_resume_company_id",
                    "_resume_project_id"
                ],
                keep="first"
            )
        )


        print(
            "기존 처리 완료 행 수:",
            len(
                previous_done
            )
        )


        tmp = tmp.merge(

            previous_done,

            on=[
                "_resume_company_id",
                "_resume_project_id"
            ],

            how="left",

            suffixes=(
                "",
                "_previous"
            )
        )


        previous_mask = (
            tmp[
                "rag_company_patent_previous"
            ]
            .notna()
        )


        tmp.loc[
            previous_mask,
            "rag_company_patent"
        ] = (
            tmp.loc[
                previous_mask,
                "rag_company_patent_previous"
            ]
        )


        tmp = tmp.drop(
            columns=[
                "rag_company_patent_previous"
            ]
        )


        print(
            "기존 결과 복원 완료"
        )


else:

    print(
        "\n기존 Drive 결과 없음 → 처음부터 시작"
    )


# ============================================================
# 27. 진행 상황
# ============================================================

processed_count = (

    tmp[
        "rag_company_patent"
    ]
    .apply(
        is_already_processed
    )
    .sum()
)


total_count = len(
    tmp
)


remaining_count = (
    total_count
    -
    processed_count
)


print(
    "\n=================================================="
)

print(
    "처리 진행 상황"
)

print(
    "=================================================="
)


print(
    "전체 행:",
    total_count
)


print(
    "처리 완료:",
    processed_count
)


print(
    "남은 행:",
    remaining_count
)


print(
    "=================================================="
)


# ============================================================
# 28. 전체 처리
# ============================================================

for idx, row in tqdm(

    tmp.iterrows(),

    total=len(
        tmp
    ),

    desc="과제별 특허 RAG"
):


    # 이미 처리된 행 skip
    if is_already_processed(
        row.get(
            "rag_company_patent",
            ""
        )
    ):

        continue


    start_time = time.time()


    company_id = normalize_company_id(
        row.get(
            "company_id",
            ""
        )
    )


    company_name = clean_text(
        row.get(
            "company_name",
            ""
        )
    )


    project_id = normalize_project_id(
        row.get(
            "project_id",
            ""
        )
    )


    project_name = clean_text(
        row.get(
            "project_name",
            ""
        )
    )


    project_description = clean_text(
        row.get(
            "project_description",
            ""
        )
    )


    patent_list = row.get(
        "company_patent_li_unique"
    )


    if not isinstance(
        patent_list,
        list
    ):

        patent_list = []


    company_patent_count = len(
        patent_list
    )


    patent_less_than_5 = (
        company_patent_count
        <
        MIN_PATENT_THRESHOLD
    )


    print(
        "\n=================================================="
    )

    print(
        f"처리 시작: "
        f"{idx + 1}/{len(tmp)}"
    )

    print(
        "company_id:",
        company_id
    )

    print(
        "company_name:",
        company_name
    )

    print(
        "project_id:",
        project_id
    )

    print(
        "project_name:",
        project_name
    )

    print(
        "중복 제거 후 고유 특허 수:",
        company_patent_count
    )

    print(
        "특허 5개 미만:",
        patent_less_than_5
    )


    # ========================================================
    # 데이터 없는 경우
    # ========================================================

    if not project_description:


        print(
            "[SKIP] project_description 없음"
        )


        selected_patents = []


    elif (
        company_patent_count
        == 0
    ):


        print(
            "[SKIP] 유효 특허 없음"
        )


        selected_patents = []


    else:


        # ====================================================
        # FAISS
        # ====================================================

        candidates = retrieve_patents_with_faiss(

            project_description=
                project_description,

            patent_list=
                patent_list,

            top_k=
                TOP_K
        )


        print(
            "FAISS Retrieval 후보 수:",
            len(
                candidates
            )
        )


        # ====================================================
        # Qwen
        # ====================================================

        response = rerank_patents_with_qwen(

            project_description=
                project_description,

            candidates=
                candidates,

            company_patent_count=
                company_patent_count
        )


        llm_result = parse_qwen_json(
            response
        )


        selected_patents = build_selected_patents(

            candidates=
                candidates,

            llm_result=
                llm_result,

            company_id=
                company_id,

            company_name=
                company_name
        )


    # ========================================================
    # 최소 3개 조건 확인
    # ========================================================

    if (
        company_patent_count
        >= MIN_PATENT_THRESHOLD

        and

        len(
            selected_patents
        )
        <
        MIN_SELECT_COUNT
    ):


        print(
            "\n[경고]"
        )


        print(
            f"고유 특허 {company_patent_count}개인데 "
            f"Qwen이 {len(selected_patents)}개만 선택"
        )


    # ========================================================
    # 현재 행 결과 반영
    # ========================================================

    tmp.at[
        idx,
        "company_patent_count"
    ] = company_patent_count


    tmp.at[
        idx,
        "patent_less_than_5"
    ] = patent_less_than_5


    tmp.at[
        idx,
        "rag_company_patent"
    ] = json.dumps(
        selected_patents,
        ensure_ascii=False
    )


    # ========================================================
    # ★ 매 행마다 10개 컬럼만 Google Drive 저장
    # ========================================================

    tmp[
        SAVE_COLS
    ].to_csv(

        DRIVE_OUTPUT_PATH,

        index=False,

        encoding="utf-8-sig"
    )


    elapsed = (
        time.time()
        -
        start_time
    )


    print(
        f"[Google Drive 저장 완료] "
        f"현재 행={idx + 1}"
    )


    print(
        "최종 선택 특허 수:",
        len(
            selected_patents
        )
    )


    print(
        f"소요 시간: "
        f"{elapsed:.2f}초"
    )


# ============================================================
# 29. 전체 완료 후 최종 저장
# ============================================================

tmp[
    SAVE_COLS
].to_csv(

    DRIVE_OUTPUT_PATH,

    index=False,

    encoding="utf-8-sig"
)


print(
    "\n=================================================="
)

print(
    "전체 처리 완료"
)

print(
    "=================================================="
)


print(
    "최종 저장 컬럼:"
)

print(
    SAVE_COLS
)


print(
    "\n최종 저장 경로:"
)

print(
    DRIVE_OUTPUT_PATH
)


# ============================================================
# 30. 최종 결과 확인
# ============================================================

display(
    tmp[
        SAVE_COLS
    ].head(
        20
    )
)